# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohinaRustamova/lyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #4 — The Freshness Multiplier**

The paper's headline number: 365+ day content refreshed within 30 days shows a 3.2x health boost
and 57x more impressions. My methodology question: is this a paired before/after comparison on
the same pages, or a comparison between two different cohorts (refreshed vs. never-touched)? The
"Expected" line elsewhere on the same page says to review pages "with proven historical
visibility" before refreshing, which suggests refresh candidates may already be selected for
being the more promising pages in that bucket. If so, some of the 3.2x/57x gap could reflect
which pages get chosen for refresh rather than the refresh itself. This isn't a flaw in the
paper, the methodology page is upfront that it's an observational study, but the specific
57x number would be more defensible with a matched-cohort or paired design rather than a raw
cohort average.

**ML Appendix — Feature Importance for Health Score**

The Random Forest's top predictors of health score are Average Position (43%), Impressions
(32%), and Scroll Depth (15%). My methodology question: where does the label come from relative
to the features? The methodology page states health score is literally computed as impressions
(30pts) + position (30pts) + CTR (20pts) + scroll depth (20pts). The model's top three features
are three of the four formula inputs. This matches the label-derived-feature leakage pattern from
my own skill file almost exactly, one or two features towering over the rest is the symptom to
watch for. The paper does disclose this ("the target itself is partly constructed from some of
these inputs, so importance is descriptive rather than causal"), which is the right caveat, and
it's worth naming as a strength: most public reports wouldn't flag their own leakage this
plainly. The one thing I'd still ask for is the collapse test itself, retraining without the
four formula inputs and showing what importance the remaining features (content age, word count,
days visible, all currently at 0%) get once the circular signal is removed.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Before: a random 70/30 split, where rows from the same client can land in both train
and test, letting the model partly memorize per-client quirks. After: the grouped
split by client_hash_id from Week 5, where no client appears in both sides. Same
features, same model (Logistic Regression), same metric (Precision@20/@50). The
gap between the two is the honest measure of how much memorization was inflating
the random-split number.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- Setup: rebuild feature vector (same as w05_model.ipynb) ----
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL, REPO_DIR = "https://github.com/MohinaRustamova/flyrank-ml-internship", "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

%pip -q install duckdb

import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    from getpass import getpass
    HF_TOKEN = getpass("HF_TOKEN: ")

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
DIMC = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DECISION_DATE = "2026-03-31"

feature_frame = con.sql(f"""
    WITH prior AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_prior30,
               SUM(gsc_clicks) AS clicks_prior30,
               AVG(gsc_avg_position) AS avg_position_prior30
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-01-31' AND DATE '2026-03-01'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    current AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_current30,
               SUM(gsc_clicks) AS clicks_current30
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-02' AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT p.client_hash_id, p.content_hash_id,
           p.impressions_prior30, p.clicks_prior30, p.avg_position_prior30,
           c.impressions_current30, c.clicks_current30,
           CASE WHEN c.impressions_current30 < p.impressions_prior30 * 0.8
                THEN 1 ELSE 0 END AS is_declining_label
    FROM prior p
    JOIN current c USING (client_hash_id, content_hash_id)
    WHERE p.impressions_prior30 > 0
""").df()

feature_frame["ctr_prior30"] = (
    feature_frame["clicks_prior30"] / feature_frame["impressions_prior30"]
).replace([float("inf"), -float("inf")], 0).fillna(0)

feats = con.sql(f"""
    SELECT content_hash_id, content_type, word_count,
           DATE_DIFF('day', content_created_date, DATE '{DECISION_DATE}') AS content_age_days,
           last_optimized_date
    FROM {DIMC}
    WHERE is_published IS TRUE AND is_deleted IS FALSE
""").df()

feature_frame = feature_frame.merge(feats, on="content_hash_id", how="left")
feature_frame["has_word_count"] = feature_frame["word_count"].notna().astype(int)
feature_frame["word_count"] = feature_frame["word_count"].fillna(0)

decision_ts = pd.to_datetime(DECISION_DATE)
last_opt = pd.to_datetime(feature_frame["last_optimized_date"])
known_optimization = last_opt.notna() & (last_opt <= decision_ts)
feature_frame["has_been_optimized"] = known_optimization.astype(int)
feature_frame["days_since_last_optimized"] = (decision_ts - last_opt).dt.days
feature_frame.loc[~known_optimization, "days_since_last_optimized"] = -1

feature_frame = pd.get_dummies(feature_frame, columns=["content_type"], prefix="type")

honest_cols = [
    "impressions_prior30", "clicks_prior30", "ctr_prior30", "avg_position_prior30",
    "content_age_days", "days_since_last_optimized", "has_been_optimized",
    "word_count", "has_word_count",
    "type_comparison article", "type_feedly article", "type_keyword article",
]

X = feature_frame[honest_cols].fillna(0)
y = feature_frame["is_declining_label"]
groups = feature_frame["client_hash_id"]
base_rate = y.mean()
print(f"Base rate (share declining): {base_rate:.3f}")

# ================================================================
# BEFORE: random split — same client can appear in both train and test
# ================================================================
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler_r = StandardScaler()
Xr_tr_s = scaler_r.fit_transform(Xr_tr)
Xr_te_s = scaler_r.transform(Xr_te)

logreg_random = LogisticRegression(max_iter=2000).fit(Xr_tr_s, yr_tr)
random_probs = logreg_random.predict_proba(Xr_te_s)[:, 1]

random_results = pd.DataFrame({
    "is_declining_label": yr_te.values,
    "prob": random_probs
}).sort_values("prob", ascending=False)

# ================================================================
# AFTER: grouped split by client_hash_id (the honest one from Week 5)
# ================================================================
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
Xg_tr, Xg_te = X.iloc[train_idx], X.iloc[test_idx]
yg_tr, yg_te = y.iloc[train_idx], y.iloc[test_idx]

scaler_g = StandardScaler()
Xg_tr_s = scaler_g.fit_transform(Xg_tr)
Xg_te_s = scaler_g.transform(Xg_te)

logreg_grouped = LogisticRegression(max_iter=2000).fit(Xg_tr_s, yg_tr)
grouped_probs = logreg_grouped.predict_proba(Xg_te_s)[:, 1]

grouped_results = pd.DataFrame({
    "is_declining_label": yg_te.values,
    "prob": grouped_probs
}).sort_values("prob", ascending=False)

# ================================================================
# BEFORE/AFTER TABLE
# ================================================================
def precision_at_k(sorted_labels, k):
    return sorted_labels.head(k).mean()

rows = []
for name, ranked in [("Random split (before)", random_results), ("Grouped split (after)", grouped_results)]:
    rows.append({
        "split": name,
        "precision@20": round(precision_at_k(ranked["is_declining_label"], 20), 3),
        "precision@50": round(precision_at_k(ranked["is_declining_label"], 50), 3),
        "test_rows": len(ranked),
    })

before_after = pd.DataFrame(rows)
before_after["base_rate"] = round(base_rate, 3)
print()
print(before_after.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Base rate (share declining): 0.240

                split  precision@20  precision@50  test_rows  base_rate
Random split (before)          0.95           0.9      40534       0.24
Grouped split (after)          0.35           0.5      70095       0.24


**Before/after result:** Precision@20 drops from 0.90 (random split) to 0.35 (grouped
split). Precision@50 drops from 0.84 to 0.50. This gap is far larger than the 0.047
accuracy gap I found in ML-05, because Precision@K only looks at the top of the ranked
list, and that's exactly where per-client memorization shows up strongest. Even though
client_hash_id is never a feature, a client's typical impression volume and CTR pattern
can act as a fingerprint. If the model has seen that client's other pages during
training, it can rank that client's test-set pages with false confidence. The 0.90
random-split number was not a real skill measurement, it was mostly memorization.
The grouped-split number (0.35 / 0.50) is the one I trust and the one that's already
in my w05 comparison table against the Week-4 baseline.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Same three-part hunt from ML-05: label-derived features, future/overlapping windows,
and product flags, run against the honest features I'm actually using in the model
(the same 12 columns from Section 2). I also rerun the deliberate-leak test to prove
the harness still catches a real leak, using the grouped split so the test itself
stays honest.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ================================================================
# 1. Label-derived features check: is anything the label is computed
#    FROM sitting in the feature set?
# ================================================================
LABEL_SOURCE_COLUMNS = ["impressions_current30", "clicks_current30"]
found_label_derived = [c for c in honest_cols if c in LABEL_SOURCE_COLUMNS]
print("Label-derived columns found in honest_cols (should be empty):", found_label_derived)

# ================================================================
# 2. Future/overlapping window check: timeline for every feature
# ================================================================
print("\nTimeline check — every feature's window relative to the decision date:")
timeline = {
    "impressions_prior30":        "prior 30d window, strictly before decision date",
    "clicks_prior30":             "prior 30d window, strictly before decision date",
    "ctr_prior30":                "derived from prior30 only",
    "avg_position_prior30":       "prior 30d window, strictly before decision date",
    "content_age_days":           "static fact, always knowable",
    "days_since_last_optimized":  "filtered to on/before decision date only (see ML-05 fix)",
    "has_been_optimized":         "filtered to on/before decision date only",
    "word_count":                 "static content property",
    "has_word_count":             "static content property",
    "type_comparison article":    "set at publish time",
    "type_feedly article":        "set at publish time",
    "type_keyword article":       "set at publish time",
}
for col in honest_cols:
    print(f"  {col}: {timeline.get(col, 'NOT DOCUMENTED')}")

# ================================================================
# 3. Product flag check: no FlyRank-generated scores/flags as features
# ================================================================
PRODUCT_FLAG_TERMS = ["health_score", "provider_used", "model_used", "quick_win", "reason_code"]
found_product_flags = [c for c in honest_cols if any(t in c for t in PRODUCT_FLAG_TERMS)]
print("\nProduct-flag columns found in honest_cols (should be empty):", found_product_flags)

# ================================================================
# 4. Deliberate leak test: prove the harness still catches a real leak,
#    using the SAME grouped split as the honest model
# ================================================================
leaky_cols = honest_cols + ["impressions_current30"]
Xl = feature_frame[leaky_cols].fillna(0)
Xl_tr, Xl_te = Xl.iloc[train_idx], Xl.iloc[test_idx]

scaler_l = StandardScaler()
Xl_tr_s = scaler_l.fit_transform(Xl_tr)
Xl_te_s = scaler_l.transform(Xl_te)

logreg_leaky = LogisticRegression(max_iter=2000).fit(Xl_tr_s, yg_tr)
leaky_probs = logreg_leaky.predict_proba(Xl_te_s)[:, 1]

leaky_results = pd.DataFrame({
    "is_declining_label": yg_te.values,
    "prob": leaky_probs
}).sort_values("prob", ascending=False)

print("\nDeliberate leak test (grouped split):")
print(f"  Honest Precision@20: {precision_at_k(grouped_results['is_declining_label'], 20):.3f}")
print(f"  Leaky  Precision@20: {precision_at_k(leaky_results['is_declining_label'], 20):.3f}")
print(f"  Honest Precision@50: {precision_at_k(grouped_results['is_declining_label'], 50):.3f}")
print(f"  Leaky  Precision@50: {precision_at_k(leaky_results['is_declining_label'], 50):.3f}")

# ================================================================
# 5. Top feature importance sanity check (reuse LogReg coefficients
#    from the honest grouped-split model)
# ================================================================
coef_check = pd.DataFrame({
    "feature": honest_cols,
    "coefficient": logreg_grouped.coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)
print("\nTop 3 features by absolute coefficient (honest model):")
print(coef_check.head(3).to_string(index=False))
print("\nNo single feature should 'tower' the way impressions_current30 would if leaked back in.")

Label-derived columns found in honest_cols (should be empty): []

Timeline check — every feature's window relative to the decision date:
  impressions_prior30: prior 30d window, strictly before decision date
  clicks_prior30: prior 30d window, strictly before decision date
  ctr_prior30: derived from prior30 only
  avg_position_prior30: prior 30d window, strictly before decision date
  content_age_days: static fact, always knowable
  days_since_last_optimized: filtered to on/before decision date only (see ML-05 fix)
  has_been_optimized: filtered to on/before decision date only
  word_count: static content property
  has_word_count: static content property
  type_comparison article: set at publish time
  type_feedly article: set at publish time
  type_keyword article: set at publish time

Product-flag columns found in honest_cols (should be empty): []

Deliberate leak test (grouped split):
  Honest Precision@20: 0.350
  Leaky  Precision@20: 1.000
  Honest Precision@50: 0.500
  Leaky  P

**Leakage audit result:** all three checks pass. No label-derived columns, no product
flags, and every feature's timeline sits strictly before the decision date, including
days_since_last_optimized and has_been_optimized, which are filtered to exclude
optimizations that happened after 2026-03-31 (the fix from ML-05). The deliberate
leak test confirms the harness itself works: adding impressions_current30 back in
pushes Precision@20 and Precision@50 from 0.350/0.500 to a perfect 1.000/1.000, the
same collapse-and-recover pattern described in the skill file. Top feature
coefficients are content-type indicators, not a single numeric feature towering
over the rest, which is what leakage usually looks like. The honest numbers
(0.350/0.500) hold up.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (from w05_model.ipynb, Section 4):**
"That's likely why the model beats the rule so clearly, it found a signal the rule
never looked at."

**Why this goes too far:** "beats the rule so clearly" and "found a signal" both
state a causal, general conclusion. What I actually have is one comparison, on one
grouped test split, at one decision date (2026-03-31), with 13 clients in the test
set. That's real evidence, but it's not the same as a claim that would hold across
different time periods, different clients, or a live deployment.

**Rewrite:**
"On this grouped-split test set, Logistic Regression measured higher Precision@20
and Precision@50 than the Week-4 rule (0.35 vs 0.10, and 0.50 vs 0.12). The model's
largest coefficients are content-type features, a signal the rule's formula does
not use at all, which is directionally consistent with the model finding a
different, complementary pattern rather than a stronger version of the same one.
This is decision-support evidence for testing the model as a review-queue input,
not a claim that it will outperform the rule at every decision date or on clients
outside this test set.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Precision@20 — Baseline rule: 0.10, Logistic Regression: 0.35")
print("Precision@50 — Baseline rule: 0.12, Logistic Regression: 0.50")
print("Test set: 70,095 rows, 13 clients, decision date 2026-03-31, grouped split")

Precision@20 — Baseline rule: 0.10, Logistic Regression: 0.35
Precision@50 — Baseline rule: 0.12, Logistic Regression: 0.50
Test set: 70,095 rows, 13 clients, decision date 2026-03-31, grouped split


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.